# 🏥 MedRAX Reproduction: Multimodal Vision Evaluation on ChestAgentBench
### Reproducing "MedRAX: Medical Reasoning Agent for Chest X-ray" (ICML 2025)
This notebook performs **genuine multimodal reproduction** of the MedRAX benchmark using free Vision-Language Models (VLM).

| Component | Specification |
| :--- | :--- |
| **Benchmark** | **ChestAgentBench** (2,500 expert-curated clinical questions across 7 clinical axes from Eurorad) |
| **Vision Model** | **Google Gemini 1.5 Flash / 2.0 Flash** (Multimodal VLM with native high-res imaging support) |
| **Cost & Limits** | **100% Free** via Google AI Studio API (15 RPM, 1,500 RPD, 0 GPU VRAM needed for LLM) |
| **Agent Tools** | **CheXagent-8B VQA**, TorchXRayVision Classifier & Segmentation, Report Generator |
| **Storage** | Google Drive persistent cache (checkpoints, dataset, predictions, graphs) |


## 1. ⚙️ Hardware & GPU Environment Diagnostics
We verify the Colab runtime environment (CUDA availability, GPU model, and VRAM).
The Colab GPU is utilized 100% for hosting MedRAX's specialized local clinical tools (**CheXagent-8B**, **TorchXRayVision**).


In [ ]:
# Check GPU availability and specifications
import os
import sys
import torch

print(f"Python Version : {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Detected GPU   : {device_name} ({vram_gb:.2f} GB VRAM)")
    print("🚀 Local GPU is fully available for MedRAX specialized clinical vision tools!")
else:
    print("⚠️ WARNING: No GPU detected! Please enable GPU in Runtime -> Change runtime type.")

!nvidia-smi


## 2. 💾 Google Drive Mounting & Persistent Storage
Mount Google Drive to persist downloaded datasets, model checkpoints, and evaluation results.
Batch commits ensure that results are flushed directly to Google Drive every 10 questions.


In [ ]:
# Mount Google Drive
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/MedRAX_Thesis')

PERSISTENT_DIRS = {
    "checkpoints": DRIVE_ROOT / "checkpoints",
    "data": DRIVE_ROOT / "data" / "chestagentbench",
    "results": DRIVE_ROOT / "results",
    "logs": DRIVE_ROOT / "logs",
}

for name, path in PERSISTENT_DIRS.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"📁 Verified persistent directory: {path}")

# Set environment variables
os.environ["MEDRAX_MODEL_DIR"] = str(PERSISTENT_DIRS["checkpoints"])
os.environ["BENCHMARK_DATA_DIR"] = str(PERSISTENT_DIRS["data"])
os.environ["MEDRAX_RESULTS_DIR"] = str(PERSISTENT_DIRS["results"])
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

print("
✅ Environment variables configured successfully.")


## 3. 🐙 Clone Reproduction Repository from GitHub
Clone the reproduction suite from GitHub. If already present, pull the latest code updates.


In [ ]:
# Configure GitHub repository
import os
import shutil

REPO_URL = "https://github.com/Uyhuynh23/MedRAX-ChestAgentBench.git"
LOCAL_REPO_DIR = Path("/content/MedRAX-ChestAgentBench")

if not LOCAL_REPO_DIR.exists():
    print(f"Cloning {REPO_URL} into {LOCAL_REPO_DIR} ...")
    !git clone {REPO_URL} {LOCAL_REPO_DIR}
else:
    print(f"Repository exists. Pulling latest updates...")
    %cd {LOCAL_REPO_DIR}
    !git pull origin main

%cd {LOCAL_REPO_DIR}
!python scripts/setup_drive.py


## 4. 📦 Install Dependencies & Specialized Radiology Packages
Install PyTorch, LangChain, TorchXRayVision, Transformers, and Google Generative AI SDK.


In [ ]:
%cd /content/MedRAX-ChestAgentBench

# Install project dependencies
!pip install -q -r requirements.txt
!pip install -q torchxrayvision
!pip install -q -U langchain-google-genai google-generativeai

print("
✅ All dependencies installed successfully.")


## 5. 👁️ Configure Multimodal Vision Model (Google Gemini API - Free Tier)

To genuinely reproduce MedRAX and evaluate on **ChestAgentBench**, the model **must be able to see and read images**.

### How to get your FREE Gemini API Key (Takes 30 seconds):
1. Visit **Google AI Studio**: [https://aistudio.google.com/apikey](https://aistudio.google.com/apikey)
2. Log in with your Google account.
3. Click **"Create API key"** and copy the key string.
4. **Cost**: 100% Free forever (15 Requests/Min, 1,500 Requests/Day). No credit card required!

> **Why this reproduces MedRAX accurately:**
> * In the original ICML 2025 MedRAX paper, the authors used **GPT-4o**. Gemini 1.5 Flash / 2.0 Flash is a frontier multimodal model of the same caliber, natively processing high-res radiographs.
> * It executes via Google Cloud API, consuming **0 MB of Colab GPU VRAM**, leaving 100% of your GPU free to run local clinical tools (CheXagent-8B and TorchXRayVision).


In [ ]:
# -------------------------------------------------------------
# Setup Free Gemini Vision API Key
# -------------------------------------------------------------
import os
from getpass import getpass

# Method 1: Check Colab Secrets (Left sidebar 🔑)
try:
    from google.colab import userdata
    gemini_key = userdata.get('GOOGLE_API_KEY') or userdata.get('GEMINI_API_KEY')
except Exception:
    gemini_key = None

# Method 2: Check Environment or Prompt
if not gemini_key:
    gemini_key = os.getenv('GOOGLE_API_KEY') or os.getenv('GEMINI_API_KEY')

if not gemini_key:
    print("🔑 Please paste your Google Gemini API Key below (from https://aistudio.google.com/apikey):")
    gemini_key = getpass("Enter GEMINI_API_KEY: ").strip()

if gemini_key:
    os.environ['GOOGLE_API_KEY'] = gemini_key
    os.environ['GEMINI_API_KEY'] = gemini_key
    print("✅ GOOGLE_API_KEY is configured and active.")
else:
    print("❌ No API key provided! Gemini Vision calls will fail without a valid key.")


In [ ]:
# -------------------------------------------------------------
# 🔍 Vision Diagnostic: Verify that Gemini can READ Chest Images
# -------------------------------------------------------------
import google.generativeai as genai
from PIL import Image
import numpy as np

genai.configure(api_key=os.environ['GOOGLE_API_KEY'])

# Create a test chest X-ray synthetic sample to verify vision capability
test_img = Image.fromarray(np.random.randint(50, 200, (256, 256, 3), dtype=np.uint8))
model = genai.GenerativeModel('gemini-1.5-flash')

response = model.generate_content([
    "Briefly describe what kind of image input this is and confirm that you can process visual data.",
    test_img
])

print("🤖 Gemini Vision Response:")
print(response.text)
print("
🎉 Vision diagnostic passed! The model can read images directly.")


## 6. 📥 Download ChestAgentBench Dataset & Extract Figures
Download the **ChestAgentBench** benchmark dataset (`wanglab/chest-agent-bench`) from Hugging Face into Google Drive.
If already present on Google Drive, it will verify integrity and skip re-downloading.


In [ ]:
# Download / Verify ChestAgentBench in persistent Google Drive
%cd /content/MedRAX-ChestAgentBench
!python scripts/download_benchmark.py --data-dir "$BENCHMARK_DATA_DIR"

!ls -lh "$BENCHMARK_DATA_DIR"
!head -n 2 "$BENCHMARK_DATA_DIR/metadata.jsonl" | cut -c 1-120


## 7. 🔍 Interactive Dataset Exploration
Inspect a sample clinical question, options, ground truth answer, and view the chest radiograph.


In [ ]:
import json
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

metadata_file = Path(os.environ["BENCHMARK_DATA_DIR"]) / "metadata.jsonl"
figures_dir = Path(os.environ["BENCHMARK_DATA_DIR"]) / "figures"

with open(metadata_file, "r", encoding="utf-8") as f:
    first_case = json.loads(f.readline())

print("📋 Case ID    :", first_case.get("case_id"))
print("❓ Question ID:", first_case.get("question_id"))
print("🏷️ Category   :", first_case.get("category") or first_case.get("categories"))
print("🎯 Gold Answer:", first_case.get("answer"))
print("
📝 Question:")
print(first_case.get("question")[:400] + "...")

images = first_case.get("images", [])
if isinstance(images, str):
    images = [images]

if images:
    sample_img_rel = images[0].replace("figures/", "") if images[0].startswith("figures/") else images[0]
    sample_img_path = figures_dir / sample_img_rel
    if sample_img_path.exists():
        img = Image.open(sample_img_path)
        plt.figure(figsize=(5, 5))
        plt.imshow(img, cmap="gray")
        plt.title(f"Case {first_case.get('case_id')} - {sample_img_rel}")
        plt.axis("off")
        plt.show()


## 8. 🧪 Benchmark Execution & Evaluation (Gemini Multimodal)

Now we evaluate on **ChestAgentBench** with Gemini Vision:
1. **Smoke Test**: Fast 2-case verification (~1 minute).
2. **Option A (Baseline Multimodal)**: Direct VLM inference where Gemini reads the radiographs natively and answers the questions.
3. **Option B (MedRAX Agent Mode)**: Gemini acts as the master clinical agent, reading the images AND coordinating specialized tools (**CheXagent-8B VQA**, TorchXRayVision classification & segmentation).


In [ ]:
# -------------------------------------------------------------
# 🧪 Step 8.1: Smoke Test - Verify Pipeline on 2 Cases (~1 min)
# -------------------------------------------------------------
%cd /content/MedRAX-ChestAgentBench

!python scripts/run_evaluation.py     --mode baseline     --model gemini-1.5-flash     --data-dir "$BENCHMARK_DATA_DIR"     --output-dir "$MEDRAX_RESULTS_DIR/smoke_test_gemini"     --max-cases 2     --reset


In [ ]:
# -------------------------------------------------------------
# 📊 Option A: Full Multimodal Baseline Evaluation (All 2,500 Questions)
# -------------------------------------------------------------
# Gemini reads high-resolution radiographs directly with native vision encoder.
# Uses batch commits every 10 questions with fsync directly to Google Drive.
%cd /content/MedRAX-ChestAgentBench

!python scripts/run_evaluation.py     --mode baseline     --model gemini-1.5-flash     --data-dir "$BENCHMARK_DATA_DIR"     --output-dir "$MEDRAX_RESULTS_DIR/gemini_baseline_full"     # Remove --reset if resuming an interrupted session
    --reset


In [ ]:
# -------------------------------------------------------------
# 🏥 Option B: Full MedRAX Agent Evaluation (All 2,500 Questions)
# -------------------------------------------------------------
# Gemini coordinates local tools (CheXagent-8B, TorchXRayVision) + Vision.
%cd /content/MedRAX-ChestAgentBench

!python scripts/run_evaluation.py     --mode medrax     --model gemini-1.5-flash     --data-dir "$BENCHMARK_DATA_DIR"     --output-dir "$MEDRAX_RESULTS_DIR/gemini_medrax_full"     # Remove --reset if resuming an interrupted session
    --reset


## 9. 📊 Comparative Analytics: Baseline vs MedRAX Agent
Generate side-by-side comparison tables and accuracy breakdown charts across the 7 clinical axes:
* **Detection**
* **Classification**
* **Localization**
* **Comparison**
* **Relationship**
* **Diagnosis**
* **Characterization**


In [ ]:
# -------------------------------------------------------------
# ⚖️ Compare Baseline vs MedRAX Agent Performance
# -------------------------------------------------------------
%cd /content/MedRAX-ChestAgentBench

!python experiments/compare_runs.py     "$MEDRAX_RESULTS_DIR/gemini_baseline_full/predictions.jsonl"     "$MEDRAX_RESULTS_DIR/gemini_medrax_full/predictions.jsonl"     --output-dir "$MEDRAX_RESULTS_DIR/comparisons"

# Display generated comparison plot
from IPython.display import Image, display
chart_path = Path(os.environ["MEDRAX_RESULTS_DIR"]) / "comparisons" / "accuracy_breakdown.png"
if chart_path.exists():
    display(Image(str(chart_path)))


## 10. 🔄 Synchronize Results & Updates to GitHub
Push any notebook changes or evaluation logs back to your GitHub repository.


In [ ]:
# Configure Git credentials in Colab
!git config --global user.name "Uy Huynh"
!git config --global user.email "huynhtanuy123@gmail.com"

# Check Git status
%cd /content/MedRAX-ChestAgentBench
!git status -s

# Uncomment below to commit & push changes:
# !git add .
# !git commit -m "docs: evaluate multimodal Gemini Vision on ChestAgentBench"
# !git push origin main
